# Another Useful Hook

Beyond `PreToolUse` and `PostToolUse`, Claude Code fires hooks on several other events.

## The other hook events

| Event | Fires when |
|-------|-----------|
| **Notification** | Claude Code sends a notification — i.e. it needs permission to use a tool, or has been idle for 60 seconds |
| **Stop** | Claude Code has finished responding |
| **SubagentStop** | a subagent (shown as a "Task" in the UI) has finished |
| **PreCompact** | just before a compact operation (manual or automatic) |
| **UserPromptSubmit** | the user submits a prompt, *before* Claude processes it |
| **SessionStart** | starting or resuming a session |
| **SessionEnd** | a session ends |

## The confusing part

The **stdin payload changes shape based on the hook type**, and — for `PreToolUse`/`PostToolUse` — the **`tool_input` changes based on which tool** was called. So you often won't know the exact structure your command receives.

**Example — `PostToolUse` watching the `TodoWrite` tool** (the tool Claude uses to track to-do items):
```json
{
  "session_id": "9ecf22fa-...",
  "transcript_path": "<path_to_transcript>",
  "hook_event_name": "PostToolUse",
  "tool_name": "TodoWrite",
  "tool_input": {
    "todos": [{ "content": "write a readme", "status": "pending", "id": "1" }]
  },
  "tool_response": {
    "oldTodos": [],
    "newTodos": [{ "content": "write a readme", "status": "pending", "id": "1" }]
  }
}
```

**For comparison — a `Stop` hook** (much smaller, no tool fields):
```json
{
  "session_id": "af9f50b6-...",
  "transcript_path": "<path_to_transcript>",
  "hook_event_name": "Stop",
  "stop_hook_active": false
}
```

## The trick: a logging helper hook

Since the input shape varies, add a **catch-all logging hook** that dumps whatever it receives to a file so you can inspect it:

```json
"PostToolUse": [   // or "PreToolUse" or "Stop", etc.
  {
    "matcher": "*",
    "hooks": [
      { "type": "command", "command": "jq . > post-log.json" }
    ]
  }
]
```

`jq .` pretty-prints the stdin JSON into `post-log.json` — now you can see **exactly** what your real command would get, and write it against the actual structure.

> This is precisely what the `queries` project does: its `settings.example.json` wires `jq . > pre-log.json` (PreToolUse) and `jq . > post-log.json` (PostToolUse). The `queries-complete` folder even ships captured `pre-log.json` / `post-log.json` samples — inspect those to see real payloads. (Requires `jq` installed.)

Next: **The Claude Code SDK** — driving Claude Code programmatically (the same mechanism the query-duplication hook used).